# 数据管道（价格预测版）

本 notebook 统一从 `data/processed/` 目录下读取 **分钟级聚合后的 CSV**，构造用于小模型与时序大模型的输入：

- 输入特征：由 `trade_volume, best_ask, best_bid, ask_size, bid_size` 等 5 元特征衍生出的微观结构特征；
- 预测目标：**未来 horizon 分钟之后的中间价 `mid_price`（价格预测）**；
- 输出格式：
  - `X_seq`: 形状为 `(N, lookback, F)` 的时间序列特征（供 LSTM/TCN/PatchTST/Informer/Lag-Llama 等使用）；
  - `X_flat`: 形状为 `(N, lookback * F)` 的平铺特征（供 LightGBM/XGBoost 等表格模型使用）；
  - `y_price`: 目标价格（未来第 `horizon` 分钟的 `mid_price`）。

> 约定：
> - 预处理后的 CSV 放在 `data/processed/`，命名为 `{ticker}_1min.csv`；
> - 每个 CSV 至少包含列：`timestamp, trade_volume, best_ask, best_bid, ask_size, bid_size`。

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Tuple


class LOBDataPipeline:
    """统一数据处理管道（价格预测版）

    输入：data_dir 下的 1 分钟级 CSV（如 {ticker}_1min.csv）
    输出：
      - X_seq:  (N, lookback, F)
      - X_flat: (N, lookback * F)
      - y:      (N,) 未来 horizon 分钟之后的 mid_price
    """

    def __init__(self, data_dir: str, lookback: int = 60, horizon: int = 30):
        self.data_dir = Path(data_dir)
        self.lookback = lookback
        self.horizon = horizon

    # --------------------------------------------------
    # 1. 读取 1 分钟级数据
    # --------------------------------------------------
    def load_raw(self, ticker: str) -> pd.DataFrame:
        path = self.data_dir / f"{ticker}_1min.csv"
        df = pd.read_csv(path, parse_dates=["timestamp"])
        df = df.set_index("timestamp").sort_index()
        df = df.between_time("09:45", "15:45")
        return df

    # --------------------------------------------------
    # 2. 特征工程
    # --------------------------------------------------
    def engineer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df["mid_price"] = (df["best_ask"] + df["best_bid"]) / 2
        df["spread"] = df["best_ask"] - df["best_bid"]
        df["order_imbalance"] = (
            (df["bid_size"] - df["ask_size"]) /
            (df["bid_size"] + df["ask_size"])
        )
        df["log_return"] = np.log(df["mid_price"]).diff()

        for w in [5, 10, 20]:
            df[f"vol_{w}"] = df["log_return"].rolling(w).std()

        for w in [5, 10, 20]:
            df[f"vol_ma_{w}"] = df["trade_volume"].rolling(w).mean()
        df["vol_spike"] = df["trade_volume"] / df["vol_ma_20"]

        for feat in ["spread", "order_imbalance", "log_return", "trade_volume"]:
            for lag in [1, 5, 10, 20]:
                df[f"{feat}_lag{lag}"] = df[feat].shift(lag)

        return df.dropna()

    # --------------------------------------------------
    # 3. 构造价格预测标签
    # --------------------------------------------------
    def build_price_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """target_price = 未来 horizon 分钟的 mid_price"""
        df = df.copy()
        df["target_price"] = df["mid_price"].shift(-self.horizon)
        df = df.iloc[:-self.horizon]
        print("target_price 统计:", df["target_price"].describe().to_dict())
        return df

    # --------------------------------------------------
    # 4. 构造滑动窗口序列
    # --------------------------------------------------
    def build_sequences(
        self,
        df: pd.DataFrame,
        feature_cols: List[str],
        target_col: str = "target_price",
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        X_list, y_list = [], []
        values = df[feature_cols].values
        labels = df[target_col].values

        for i in range(self.lookback, len(df)):
            X_list.append(values[i - self.lookback: i])
            y_list.append(labels[i])

        X_seq = np.array(X_list)
        X_flat = X_seq.reshape(len(X_seq), -1)
        y = np.array(y_list)
        return X_seq, X_flat, y

    # --------------------------------------------------
    # 5. 滚动前向验证切分
    # --------------------------------------------------
    def walk_forward_split(
        self,
        df: pd.DataFrame,
        train_days: int = 63,
        val_days: int = 21,
        step_days: int = 5,
    ):
        daily_minutes = 360
        train_size = train_days * daily_minutes
        val_size = val_days * daily_minutes
        step_size = step_days * daily_minutes

        start = 0
        while start + train_size + val_size <= len(df):
            train_idx = slice(start, start + train_size)
            val_idx = slice(start + train_size, start + train_size + val_size)
            yield train_idx, val_idx
            start += step_size


print("LOBDataPipeline 已定义。")

## 使用示例

当 `data/processed/` 下有对应的 `{ticker}_1min.csv` 时，取消注释运行以下代码：

In [ ]:
# pipeline = LOBDataPipeline(data_dir="processed", lookback=60, horizon=30)
# ticker = "1319"
#
# df_raw = pipeline.load_raw(ticker)
# print("原始行数:", len(df_raw))
#
# df_feat = pipeline.engineer_features(df_raw)
# print("特征工程后行数:", len(df_feat))
#
# df_labeled = pipeline.build_price_labels(df_feat)
# print("标签构建后行数:", len(df_labeled))
#
# feature_cols = [
#     "mid_price", "spread", "order_imbalance", "log_return",
#     "vol_5", "vol_10", "vol_20",
#     "vol_ma_5", "vol_ma_10", "vol_ma_20", "vol_spike",
# ]
# X_seq, X_flat, y = pipeline.build_sequences(df_labeled, feature_cols)
# print("X_seq:", X_seq.shape, "X_flat:", X_flat.shape, "y:", y.shape)